# Notebook 00: Environment Setup & API Authentication
This notebook verifies that all required Python packages are installed and details the authentication procedures for Google Earth Engine (GEE) and Copernicus Data Space Ecosystem (CDSE).

In [ ]:
import sys
print(f"Python Version: {sys.version}")

# Check core geospatial dependencies
libs = ["ee", "sentinelhub", "geopandas", "rasterio", "sklearn", "numpy", "pandas"]
for lib in libs:
    try:
        mod = __import__(lib)
        version = getattr(mod, "__version__", "Installed")
        print(f"✓ {lib}: {version}")
    except ImportError:
        print(f"✗ {lib}: Missing! Run 'pip install -r ../requirements.txt'")

## Google Earth Engine Authentication
To interact with Google Earth Engine, you must link your Google Account and register a Cloud Project. Run the cell below to trigger the OAuth login flow.

In [ ]:
import os
import json
import ee
from dotenv import load_dotenv

load_dotenv()

# Check for local client_secret.json to configure custom OAuth client
client_secret_path = os.path.abspath(os.path.join(os.getcwd(), "..", "client_secret.json"))
if not os.path.exists(client_secret_path):
    # Try current directory too just in case
    client_secret_path = os.path.abspath("client_secret.json")

if os.path.exists(client_secret_path):
    try:
        with open(client_secret_path, "r", encoding="utf-8") as f:
            secret_data = json.load(f)
        installed_info = secret_data.get("installed", {})
        if installed_info:
            client_id = installed_info.get("client_id")
            client_secret_val = installed_info.get("client_secret")
            if client_id and client_secret_val:
                print("Configuring Earth Engine custom client credentials from client_secret.json...")
                ee.oauth.CLIENT_ID = client_id
                ee.oauth.CLIENT_SECRET = client_secret_val
    except Exception as e:
        print(f"Failed to load client_secret.json: {e}")

try:
    # Trigger GEE authentication flow
    ee.Authenticate()
    print("Authentication successful.")
except Exception as e:
    print(f"Authentication failed: {e}")

## Initialize GEE API Connection
Once authenticated, initialize Earth Engine under your designated project ID.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Use project from env var
project_id = os.getenv("GCP_PROJECT", "tesis-500804")

try:
    import ee
    ee.Initialize(project=project_id)
    print(f"Earth Engine successfully initialized with project: {project_id}")
except Exception as e:
    print(f"Earth Engine initialization failed: {e}")